<table align="left">
  <td style="text-align: center">
    <a href="https://colab.research.google.com/github/arvind-dhariwal/aeon-credit-gcp-workshop/blob/main/track1_platform_governance/02_Data_Engineering_Agent_Medallion_Transformation.ipynb">
      <img width="32px" src="https://www.gstatic.com/pantheon/images/bigquery/welcome_page/colab-logo.svg" alt="Google Colaboratory logo"><br> Open in Colab
    </a>
  </td>
  <td style="text-align: center">
    <a href="https://console.cloud.google.com/vertex-ai/colab/import/https:%2F%2Fraw.githubusercontent.com%2Farvind-dhariwal%2Faeon-credit-gcp-workshop%2Fmain%2Ftrack1_platform_governance%2F02_Data_Engineering_Agent_Medallion_Transformation.ipynb">
      <img width="32px" src="https://lh3.googleusercontent.com/JmcxdQi-qOpctIvWKgPtrzZdJJK-J3sWE1RsfjZNwshCFgE_9fULcNpuXYTilIR2hjwN" alt="Google Cloud Colab Enterprise logo"><br> Open in Colab Enterprise
    </a>
  </td>
  <td style="text-align: center">
    <a href="https://console.cloud.google.com/vertex-ai/workbench/deploy-notebook?download_url=https://raw.githubusercontent.com/arvind-dhariwal/aeon-credit-gcp-workshop/main/track1_platform_governance/02_Data_Engineering_Agent_Medallion_Transformation.ipynb">
      <img width="32px" src="https://storage.googleapis.com/github-repo/workbench-icon.svg" alt="Workbench logo"><br> Open in Workbench
    </a>
  </td>
  <td style="text-align: center">
    <a href="https://console.cloud.google.com/bigquery/import?url=https://github.com/arvind-dhariwal/aeon-credit-gcp-workshop/blob/main/track1_platform_governance/02_Data_Engineering_Agent_Medallion_Transformation.ipynb">
      <img src="https://www.gstatic.com/images/branding/gcpiconscolors/bigquery/v1/32px.svg" alt="BigQuery Studio logo"><br> Open in BigQuery Studio
    </a>
  </td>
  <td style="text-align: center">
    <a href="https://github.com/arvind-dhariwal/aeon-credit-gcp-workshop/blob/main/track1_platform_governance/02_Data_Engineering_Agent_Medallion_Transformation.ipynb">
      <img width="32px" src="https://raw.githubusercontent.com/primer/octicons/refs/heads/main/icons/mark-github-24.svg" alt="GitHub logo"><br> View on GitHub
    </a>
  </td>
</table>

<div style="clear: both;"></div>

---

# Track 1 (Notebook 2): Agentic Medallion Transformation (`acsm_bronze` → `acsm_silver` → `acsm_gold`)
**AEON Credit Service Malaysia (ACSM) — Google Agentic Data Cloud Workshop (`asia-southeast1` Singapore)**

---

## 🗺️ Notebook 2 Architecture & Medallion DAG (`acsm_bronze` → `acsm_silver` → `acsm_gold`)

![Track 1 Notebook 2 — Agentic Medallion Transformation Flow](https://raw.githubusercontent.com/arvind-dhariwal/aeon-credit-gcp-workshop/main/track1_platform_governance/images/notebook2_medallion_architecture_flow.png)

---
### 📋 Notebook 2 Step-by-Step Execution Summary
1. **Step 1**: Configure parameterized `PROJECT_ID` (auto-detects active project if blank) & `LOCATION = "asia-southeast1"` (Singapore) and enable the Data Engineering Agent APIs (`dataform.googleapis.com`, `cloudaicompanion.googleapis.com`).
2. **Step 2 (`%%bigquery` SQL)**: Verify all **8 `acsm_bronze` source tables** (`1,398,284` rows) are ready.
3. **Step 3 (`%%bigquery` SQL)**: Create the two target Medallion datasets — **`acsm_silver`** and **`acsm_gold`** — in **`asia-southeast1` (Singapore)**.
4. **Step 4 (UI Guide + Prompt)**: Exact click-by-click instructions on **where and how to copy-paste the Medallion Architecture Prompt** into the **BigQuery Data Engineering Agent (`+ Create` → `Pipeline`)**.
5. **Step 5 (`%%bigquery` SQL)**: Transparent SQL execution of the exact same **4 Silver tables + 1 Gold AEON 360 table** directly inside this notebook.
6. **Step 6 (`%%bigquery` SQL)**: Verify `acsm_silver` & `acsm_gold` row counts and run the **Dual-Run Financial Control Total Reconciliation Audit (`0.00 MYR Variance`)**.


---
## Step 1: Configure Project & Region (`asia-southeast1`) and Enable Data Engineering Agent APIs
Run the cell below to set your `PROJECT_ID` and `LOCATION` (`asia-southeast1` Singapore), load the `%bigquery` SQL magic extension, and enable the required APIs (`dataform.googleapis.com` and `cloudaicompanion.googleapis.com`) for the **BigQuery Data Engineering Agent**.

In [ ]:
# @title 1. Set Parameterized `PROJECT_ID` (Auto-Detects Active Project if Blank), Singapore Region (`asia-southeast1`), and Enable APIs
import os
import subprocess

PROJECT_ID = ""  # @param {type:"string"}
if not PROJECT_ID or PROJECT_ID.startswith("<"):
    PROJECT_ID = (
        os.environ.get("GOOGLE_CLOUD_PROJECT")
        or subprocess.check_output(["gcloud", "config", "get-value", "project"], text=True).strip()
    )
LOCATION = "asia-southeast1"  # @param {type:"string"}

os.environ["PROJECT_ID"] = PROJECT_ID
os.environ["LOCATION"] = LOCATION

# Load the native BigQuery SQL magic so all subsequent cells run pure SQL against $PROJECT_ID
%load_ext google.cloud.bigquery

# Enable BigQuery Pipelines (Dataform) and Gemini for Google Cloud (Data Engineering Agent)
!gcloud config set project $PROJECT_ID
!gcloud services enable bigquery.googleapis.com dataform.googleapis.com cloudaicompanion.googleapis.com --project=$PROJECT_ID
print(f"Ready! Active Project={PROJECT_ID} | Region={LOCATION} (Singapore)")


---
## Step 2: Pre-Flight Check — Verify the 8 Source Tables in `acsm_bronze` (`%%bigquery`)
Before creating the Silver and Gold Medallion layers, run the `%%bigquery` SQL cell below to verify that all **8 core ACSM tables** (`Fact_EP_Judge`, `Fact_EP_Sales`, `Fact_EP_Collection`, `Fact_CC_Judge`, `Fact_CC_Sales`, `Fact_CC_Collection`, `m3CIF`, `dimProduct`) are loaded in `acsm_bronze`.

In [ ]:
%%bigquery --project $PROJECT_ID --location $LOCATION
-- Verify all 8 source tables & views in acsm_bronze (6 Native Fact Tables + m3CIF Lakehouse Iceberg View + dimProduct AWS Glue Federated Iceberg View)
SELECT 'Fact_EP_Judge'      AS table_name, 'BigQuery Native Table'                AS storage_engine, COUNT(*) AS bronze_row_count FROM `acsm_bronze.Fact_EP_Judge`
UNION ALL
SELECT 'Fact_EP_Sales'      AS table_name, 'BigQuery Native Table'                AS storage_engine, COUNT(*) AS bronze_row_count FROM `acsm_bronze.Fact_EP_Sales`
UNION ALL
SELECT 'Fact_EP_Collection' AS table_name, 'BigQuery Native Table'                AS storage_engine, COUNT(*) AS bronze_row_count FROM `acsm_bronze.Fact_EP_Collection`
UNION ALL
SELECT 'Fact_CC_Judge'      AS table_name, 'BigQuery Native Table'                AS storage_engine, COUNT(*) AS bronze_row_count FROM `acsm_bronze.Fact_CC_Judge`
UNION ALL
SELECT 'Fact_CC_Sales'      AS table_name, 'BigQuery Native Table'                AS storage_engine, COUNT(*) AS bronze_row_count FROM `acsm_bronze.Fact_CC_Sales`
UNION ALL
SELECT 'Fact_CC_Collection' AS table_name, 'BigQuery Native Table'                AS storage_engine, COUNT(*) AS bronze_row_count FROM `acsm_bronze.Fact_CC_Collection`
UNION ALL
SELECT 'm3CIF'              AS table_name, 'GCP Lakehouse Iceberg View (GCS)'     AS storage_engine, COUNT(*) AS bronze_row_count FROM `acsm_bronze.m3CIF`
UNION ALL
SELECT 'dimProduct'         AS table_name, 'AWS Glue Federated Iceberg View (S3)' AS storage_engine, COUNT(*) AS bronze_row_count FROM `acsm_bronze.dimProduct`
ORDER BY table_name;

---
## Step 3: Create the Target Medallion Datasets (`acsm_silver` & `acsm_gold`) in Singapore (`asia-southeast1`)
The **BigQuery Data Engineering Agent** requires the target datasets (`acsm_silver` and `acsm_gold`) to exist in the same region (`asia-southeast1` Singapore) as `acsm_bronze` before generating and running the pipeline.

Run the `%%bigquery` SQL cell below to create both datasets and verify all 3 Medallion schemas (`acsm_bronze`, `acsm_silver`, `acsm_gold`).

In [ ]:
%%bigquery --project $PROJECT_ID --location $LOCATION
-- 1. Create the Silver Medallion Dataset in Singapore (asia-southeast1)
CREATE SCHEMA IF NOT EXISTS `acsm_silver`
OPTIONS (
  location = 'asia-southeast1',
  description = 'ACSM Silver Medallion Layer: Standardized, type-cast, and deduplicated Customer CIF, EP/CC Underwriting, and Collections tables in Singapore (asia-southeast1)'
);

-- 2. Create the Gold Medallion Dataset in Singapore (asia-southeast1)
CREATE SCHEMA IF NOT EXISTS `acsm_gold`
OPTIONS (
  location = 'asia-southeast1',
  description = 'ACSM Gold Medallion Layer: Unified AEON 360 Customer Risk, Affordability & Credit Exposure Feature Store in Singapore (asia-southeast1)'
);

-- 3. Verify all 3 Medallion Datasets (Bronze, Silver, Gold) in INFORMATION_SCHEMA
SELECT
  schema_name AS dataset_name,
  location,
  creation_time
FROM `region-asia-southeast1`.INFORMATION_SCHEMA.SCHEMATA
WHERE schema_name IN ('acsm_bronze', 'acsm_silver', 'acsm_gold')
ORDER BY schema_name;

---
## Step 4: How & Where to Copy-Paste the Medallion Architecture Prompt in the BigQuery Data Engineering Agent

Now that `acsm_bronze`, `acsm_silver`, and `acsm_gold` are ready in **`asia-southeast1` (Singapore)**, follow these exact clicks in **BigQuery Studio** to generate your **5-node Medallion Pipeline DAG** using natural language:

### 🧭 Part A: Where to Click in BigQuery Studio
1. Open **Google Cloud Console** $\rightarrow$ navigate to **BigQuery** $\rightarrow$ **Studio**.
2. In the left **Explorer** pane, expand your project (`${PROJECT_ID}`) and verify you see the three datasets:
   - `acsm_bronze` *(contains the 8 ingested tables)*
   - `acsm_silver` *(created in Step 3)*
   - `acsm_gold` *(created in Step 3)*
3. At the top of the **BigQuery Studio workspace tab bar** (next to **`+ SQL query`** and **`+ Notebook`**), click **`+` (Create new)** $\rightarrow$ select **`Pipeline`**.
4. When prompted for the **Pipeline Location / Code Region**, select **`asia-southeast1 (Singapore)`** (this must match the region of `acsm_bronze`, `acsm_silver`, and `acsm_gold`).
   - *Tip*: If BigQuery displays a banner asking to grant the **Dataform Service Agent** (`service-<PROJECT_NUMBER>@gcp-sa-dataform.iam.gserviceaccount.com`) access to BigQuery, click **`Grant`** (grants `roles/bigquery.dataEditor` and `roles/bigquery.jobUser`).
5. Inside the **Pipeline Canvas**, click the **Gemini Sparkle button (`Ask Data Engineering Agent` / `Generate with Gemini`)** in the canvas toolbar or bottom prompt bar.

---
### 📋 Part B: Copy & Paste This Exact Medallion Architecture Prompt
Copy the entire prompt block below and paste it into the **Data Engineering Agent** prompt box, then click **Generate**:

```text
Build a Medallion architecture data transformation pipeline in BigQuery (location asia-southeast1) using source tables from the `acsm_bronze` dataset to populate standardized tables in the `acsm_silver` dataset and an executive Customer 360 feature store in the `acsm_gold` dataset:

1. Create Silver table `acsm_silver.silver_customer_cif` from `acsm_bronze.m3CIF`:
   - Deduplicate records by `CIF_ID` keeping the latest record based on `Rcd_DT`.
   - Select `CIF_ID`, `CIF_NM`, `Gender`, `MaritalSts`, `Citizen`, `State`, `Region`, `Race`, `Occupation`, `EmpSts`, `N_Age`, `N_YrStay`, `N_YrJob`,
     CAST(`B_NetIncome` AS NUMERIC) AS `B_NetIncome`,
     CAST(`B_GrossIncome` AS NUMERIC) AS `B_GrossIncome`,
     CAST(`B_AnnualIncome` AS NUMERIC) AS `B_AnnualIncome`,
     and `RecvPromo_FG`.

2. Create Silver table `acsm_silver.silver_ep_underwriting` from `acsm_bronze.Fact_EP_Judge`:
   - Standardize `CIF_NO` as `CIF_ID`.
   - Parse integer date `APPL_DT` (format YYYYMMDD) into a BigQuery DATE column `application_date` using SAFE.PARSE_DATE('%Y%m%d', CAST(APPL_DT AS STRING)).
   - Include `APPL_NO`, `AGREE_NO`, `CIF_ID`, `application_date`, `APPL_STS`, `SCORING_POINT`, `SCORING_RANK`, `SCORE_DECISION`, `LOAN_GRP`, `FIN_AMT`, `INST_AMT`, `INTEREST`, `TOTAL_INST`, `NetIncome`, `NDI`, `CUR_DSR`, `NEW_DSR`, and `TOTAL_AEON_OSB`.

3. Create Silver table `acsm_silver.silver_cc_underwriting` from `acsm_bronze.Fact_CC_Judge`:
   - Parse integer date `Appl_DT` (format YYYYMMDD) into a BigQuery DATE column `application_date` using SAFE.PARSE_DATE('%Y%m%d', CAST(Appl_DT AS STRING)).
   - Include `Appl_ID`, `Account_No`, `CIF_ID`, `application_date`, `ApplSts_ID`, `CardTyp_ID`, `CardBrand_ID`, `ScoreDecision_ID`, `ScoreRank_ID`, `NetIncome`, `NDI`, `CurrDSR`, `NewDSR`, `B_CrLimit`, `Final_Score`, and `Final_ScoreDesc`.

4. Create Silver table `acsm_silver.silver_collections_summary` by combining `acsm_bronze.Fact_EP_Collection` and `acsm_bronze.Fact_CC_Collection`:
   - Aggregate at the customer level (`CIF_No` aliased as `CIF_ID`) to compute:
     `total_ep_unpaid_osp` (SUM of Unpaid_OSP from `Fact_EP_Collection`),
     `total_cc_unpaid_osp` (SUM of Unpaid_OSP from `Fact_CC_Collection`),
     `combined_unpaid_osp` (total_ep_unpaid_osp + total_cc_unpaid_osp),
     `worst_collection_score_grade` (MAX of Score_Grade across both tables).

5. Create Gold table `acsm_gold.gold_aeon360_customer_profile`:
   - Join `acsm_silver.silver_customer_cif` (base customer table) with:
     a) Aggregated `acsm_silver.silver_ep_underwriting` per `CIF_ID` (total EP applications `ep_app_count`, total financed amount `total_ep_financed_myr`, average EP new DSR `avg_ep_new_dsr`),
     b) Aggregated `acsm_silver.silver_cc_underwriting` per `CIF_ID` (total CC applications `cc_app_count`, total approved credit limit `total_cc_limit_myr`, latest CTOS score `latest_ctos_score`),
     c) `acsm_silver.silver_collections_summary` per `CIF_ID` (`combined_unpaid_osp`, `worst_collection_score_grade`),
     d) Aggregated `acsm_bronze.dimProduct` per `CIF_ID` (count of active credit cards `active_card_count`, total credit purchase usage `total_cp_usage_myr`, total credit purchase available `total_cp_available_myr`).
   - Cluster `acsm_gold.gold_aeon360_customer_profile` by `State` and `CIF_ID`.
```

---
### ▶️ Part C: Review the Visual DAG & Run the Pipeline
1. The **Data Engineering Agent** will automatically generate a **5-node visual Dataform DAG** on your Pipeline Canvas:
   - **4 Silver transformation nodes** (`silver_customer_cif`, `silver_ep_underwriting`, `silver_cc_underwriting`, `silver_collections_summary`) running in parallel from `acsm_bronze`.
   - **1 downstream Gold join node** (`gold_aeon360_customer_profile`) waiting for all 4 Silver tables + `dimProduct`.
2. Click any node on the canvas to inspect the generated SQL statement.
3. Click **`Apply`** to accept the agent's generated nodes, then click **`Run`** at the top of the Pipeline Canvas to materialize all 5 Silver & Gold tables in BigQuery!

---
## Step 5: Transparent `%%bigquery` SQL Medallion Execution (Direct SQL Option & Reconciliation)
In addition to (or as a direct SQL comparison to) running the Pipeline Canvas in Step 4, you can execute the **exact same 5 Silver & Gold Medallion table transformations + Dual-Run Financial Reconciliation Audit** right here in this notebook using `%%bigquery` SQL so every line of transformation logic is 100% transparent.

In [ ]:
%%bigquery --project $PROJECT_ID --location $LOCATION
-- =============================================================================
-- 1. SILVER LAYER: acsm_silver.silver_customer_cif (Deduplicated Customer Master)
-- =============================================================================
CREATE OR REPLACE TABLE `acsm_silver.silver_customer_cif`
CLUSTER BY CIF_ID, State
OPTIONS (
  description = 'Governed Silver Customer Master (m3CIF) deduplicated by CIF_ID with typed income and PDPA consent flags.'
) AS
SELECT
  CAST(CIF_ID AS STRING) AS CIF_ID,
  SAFE.PARSE_DATE('%Y-%m-%d', SUBSTR(CAST(Rcd_DT AS STRING), 1, 10)) AS record_refresh_date,
  TRIM(CAST(CIF_NM AS STRING)) AS CIF_NM,
  TRIM(CAST(Gender AS STRING)) AS Gender,
  TRIM(CAST(MaritalSts AS STRING)) AS MaritalSts,
  TRIM(CAST(Citizen AS STRING)) AS Citizen,
  TRIM(CAST(State AS STRING)) AS State,
  TRIM(CAST(Region AS STRING)) AS Region,
  TRIM(CAST(Race AS STRING)) AS Race,
  TRIM(CAST(Occupation AS STRING)) AS Occupation,
  CAST(EmpSts AS INT64) AS EmpSts,
  CAST(N_Age AS INT64) AS N_Age,
  CAST(N_YrStay AS NUMERIC) AS N_YrStay,
  CAST(N_YrJob AS NUMERIC) AS N_YrJob,
  CAST(B_NetIncome AS NUMERIC) AS B_NetIncome,
  CAST(B_GrossIncome AS NUMERIC) AS B_GrossIncome,
  CAST(B_AnnualIncome AS NUMERIC) AS B_AnnualIncome,
  COALESCE(TRIM(CAST(RecvPromo_FG AS STRING)), 'N') AS RecvPromo_FG
FROM `acsm_bronze.m3CIF`
QUALIFY ROW_NUMBER() OVER (PARTITION BY CAST(CIF_ID AS STRING) ORDER BY Rcd_DT DESC) = 1;

-- =============================================================================
-- 2. SILVER LAYER: acsm_silver.silver_ep_underwriting (Easy Payment Underwriting)
-- =============================================================================
CREATE OR REPLACE TABLE `acsm_silver.silver_ep_underwriting`
CLUSTER BY CIF_ID, APPL_STS
OPTIONS (
  description = 'Governed Silver Easy Payment (EP) Application & Underwriting Decisions from acsm_bronze.Fact_EP_Judge.'
) AS
SELECT
  CAST(APPL_NO AS STRING) AS APPL_NO,
  CAST(AGREE_NO AS STRING) AS AGREE_NO,
  CAST(CIF_NO AS STRING) AS CIF_ID,
  SAFE.PARSE_DATE('%Y%m%d', NULLIF(TRIM(CAST(APPL_DT AS STRING)), '0')) AS application_date,
  TRIM(CAST(APPL_STS AS STRING)) AS APPL_STS,
  CAST(SCORING_POINT AS NUMERIC) AS SCORING_POINT,
  TRIM(CAST(SCORING_RANK AS STRING)) AS SCORING_RANK,
  TRIM(CAST(SCORE_DECISION AS STRING)) AS SCORE_DECISION,
  TRIM(CAST(LOAN_GRP AS STRING)) AS LOAN_GRP,
  CAST(FIN_AMT AS NUMERIC) AS FIN_AMT,
  CAST(INST_AMT AS NUMERIC) AS INST_AMT,
  CAST(INTEREST AS NUMERIC) AS INTEREST,
  CAST(TOTAL_INST AS INT64) AS TOTAL_INST,
  CAST(NetIncome AS NUMERIC) AS NetIncome,
  CAST(NDI AS NUMERIC) AS NDI,
  CAST(CUR_DSR AS NUMERIC) AS CUR_DSR,
  CAST(NEW_DSR AS NUMERIC) AS NEW_DSR,
  CAST(TOTAL_AEON_OSB AS NUMERIC) AS TOTAL_AEON_OSB
FROM `acsm_bronze.Fact_EP_Judge`;

-- =============================================================================
-- 3. SILVER LAYER: acsm_silver.silver_cc_underwriting (Credit Card Underwriting)
-- =============================================================================
CREATE OR REPLACE TABLE `acsm_silver.silver_cc_underwriting`
CLUSTER BY CIF_ID, ApplSts_ID
OPTIONS (
  description = 'Governed Silver Credit Card Application & Underwriting Decisions from acsm_bronze.Fact_CC_Judge.'
) AS
SELECT
  CAST(Appl_ID AS STRING) AS Appl_ID,
  CAST(Account_No AS STRING) AS Account_No,
  CAST(CIF_ID AS STRING) AS CIF_ID,
  SAFE.PARSE_DATE('%Y%m%d', NULLIF(TRIM(CAST(Appl_DT AS STRING)), '0')) AS application_date,
  TRIM(CAST(ApplSts_ID AS STRING)) AS ApplSts_ID,
  TRIM(CAST(CardTyp_ID AS STRING)) AS CardTyp_ID,
  TRIM(CAST(CardBrand_ID AS STRING)) AS CardBrand_ID,
  TRIM(CAST(ScoreDecision_ID AS STRING)) AS ScoreDecision_ID,
  TRIM(CAST(ScoreRank_ID AS STRING)) AS ScoreRank_ID,
  CAST(NetIncome AS NUMERIC) AS NetIncome,
  CAST(NDI AS NUMERIC) AS NDI,
  CAST(CurrDSR AS NUMERIC) AS CurrDSR,
  CAST(NewDSR AS NUMERIC) AS NewDSR,
  CAST(B_CrLimit AS NUMERIC) AS B_CrLimit,
  CAST(Final_Score AS NUMERIC) AS Final_Score,
  TRIM(CAST(Final_ScoreDesc AS STRING)) AS Final_ScoreDesc
FROM `acsm_bronze.Fact_CC_Judge`;

-- =============================================================================
-- 4. SILVER LAYER: acsm_silver.silver_collections_summary (EP + CC Collections)
-- =============================================================================
CREATE OR REPLACE TABLE `acsm_silver.silver_collections_summary`
CLUSTER BY CIF_ID
OPTIONS (
  description = 'Customer-level Silver Collections & Delinquency Summary across Fact_EP_Collection and Fact_CC_Collection.'
) AS
WITH ep_col AS (
  SELECT
    CAST(CIF_No AS STRING) AS CIF_ID,
    SUM(CAST(Unpaid_OSP AS NUMERIC)) AS total_ep_unpaid_osp,
    MAX(TRIM(CAST(Score_Grade AS STRING))) AS ep_worst_grade
  FROM `acsm_bronze.Fact_EP_Collection`
  GROUP BY 1
),
cc_col AS (
  SELECT
    CAST(CIF_No AS STRING) AS CIF_ID,
    SUM(CAST(Unpaid_OSP AS NUMERIC)) AS total_cc_unpaid_osp,
    MAX(TRIM(CAST(Score_Grade AS STRING))) AS cc_worst_grade
  FROM `acsm_bronze.Fact_CC_Collection`
  GROUP BY 1
)
SELECT
  COALESCE(ep.CIF_ID, cc.CIF_ID) AS CIF_ID,
  COALESCE(ep.total_ep_unpaid_osp, 0) AS total_ep_unpaid_osp,
  COALESCE(cc.total_cc_unpaid_osp, 0) AS total_cc_unpaid_osp,
  COALESCE(ep.total_ep_unpaid_osp, 0) + COALESCE(cc.total_cc_unpaid_osp, 0) AS combined_unpaid_osp,
  GREATEST(COALESCE(ep.ep_worst_grade, 'A'), COALESCE(cc.cc_worst_grade, 'A')) AS worst_collection_score_grade
FROM ep_col ep
FULL OUTER JOIN cc_col cc
  ON ep.CIF_ID = cc.CIF_ID;

-- =============================================================================
-- 5. GOLD LAYER: acsm_gold.gold_aeon360_customer_profile (AEON 360 Feature Store)
-- =============================================================================
CREATE OR REPLACE TABLE `acsm_gold.gold_aeon360_customer_profile`
CLUSTER BY State, CIF_ID
OPTIONS (
  description = 'Gold AEON 360 Customer Risk, Affordability & Credit Exposure Feature Store joining CIF, EP Underwriting, CC Underwriting, Collections, and Card Utilization.'
) AS
WITH ep_agg AS (
  SELECT
    CIF_ID,
    COUNT(*) AS ep_app_count,
    ROUND(SUM(COALESCE(FIN_AMT, 0)), 2) AS total_ep_financed_myr,
    ROUND(AVG(NEW_DSR), 2) AS avg_ep_new_dsr
  FROM `acsm_silver.silver_ep_underwriting`
  GROUP BY CIF_ID
),
cc_agg AS (
  SELECT
    CIF_ID,
    COUNT(*) AS cc_app_count,
    ROUND(SUM(COALESCE(B_CrLimit, 0)), 2) AS total_cc_limit_myr,
    MAX(Final_Score) AS latest_ctos_score
  FROM `acsm_silver.silver_cc_underwriting`
  GROUP BY CIF_ID
),
card_agg AS (
  SELECT
    CAST(CIF_ID AS STRING) AS CIF_ID,
    COUNTIF(TRIM(CAST(Card_Status AS STRING)) = 'Active') AS active_card_count,
    ROUND(SUM(CAST(CP_CL_Usage AS NUMERIC)), 2) AS total_cp_usage_myr,
    ROUND(SUM(CAST(CP_CL_Available AS NUMERIC)), 2) AS total_cp_available_myr
  FROM `acsm_bronze.dimProduct`
  GROUP BY 1
)
SELECT
  c.CIF_ID,
  c.CIF_NM,
  c.State,
  c.Region,
  c.Occupation,
  c.N_Age,
  c.B_NetIncome,
  c.B_AnnualIncome,
  c.RecvPromo_FG,
  COALESCE(ep.ep_app_count, 0) AS ep_app_count,
  COALESCE(ep.total_ep_financed_myr, 0) AS total_ep_financed_myr,
  ep.avg_ep_new_dsr,
  COALESCE(cc.cc_app_count, 0) AS cc_app_count,
  COALESCE(cc.total_cc_limit_myr, 0) AS total_cc_limit_myr,
  cc.latest_ctos_score,
  COALESCE(col.total_ep_unpaid_osp, 0) AS total_ep_unpaid_osp,
  COALESCE(col.total_cc_unpaid_osp, 0) AS total_cc_unpaid_osp,
  COALESCE(col.combined_unpaid_osp, 0) AS combined_unpaid_osp,
  COALESCE(col.worst_collection_score_grade, 'NONE') AS worst_collection_score_grade,
  COALESCE(crd.active_card_count, 0) AS active_card_count,
  COALESCE(crd.total_cp_usage_myr, 0) AS total_cp_usage_myr,
  COALESCE(crd.total_cp_available_myr, 0) AS total_cp_available_myr
FROM `acsm_silver.silver_customer_cif` c
LEFT JOIN ep_agg ep USING (CIF_ID)
LEFT JOIN cc_agg cc USING (CIF_ID)
LEFT JOIN `acsm_silver.silver_collections_summary` col USING (CIF_ID)
LEFT JOIN card_agg crd USING (CIF_ID);

---
## Step 6: Verify `acsm_silver` & `acsm_gold` Tables and Dual-Run Financial Reconciliation (`%%bigquery`)
Run the two `%%bigquery` SQL cells below to:
1. Inspect the created tables and row counts in `acsm_silver` and `acsm_gold`.
2. Run the **Dual-Run Financial Control Total Reconciliation Audit** (`Bronze vs. Silver/Gold`) to prove **0.00 MYR variance** across Easy Payment financed principal (`FIN_AMT`) and Collections unpaid principal (`Unpaid_OSP`).

In [ ]:
%%bigquery --project $PROJECT_ID --location $LOCATION
-- 1. Verify all created Medallion tables across acsm_silver and acsm_gold
SELECT
  table_schema AS medallion_layer,
  table_name,
  SUM(total_rows) AS total_rows,
  ROUND(SUM(total_logical_bytes) / (1024 * 1024), 2) AS logical_mb
FROM `acsm_silver.INFORMATION_SCHEMA.PARTITIONS`
GROUP BY table_schema, table_name
UNION ALL
SELECT
  table_schema AS medallion_layer,
  table_name,
  SUM(total_rows) AS total_rows,
  ROUND(SUM(total_logical_bytes) / (1024 * 1024), 2) AS logical_mb
FROM `acsm_gold.INFORMATION_SCHEMA.PARTITIONS`
GROUP BY table_schema, table_name
ORDER BY medallion_layer, table_name;

In [ ]:
%%bigquery --project $PROJECT_ID --location $LOCATION
-- 2. Dual-Run Financial Control Total Reconciliation Audit (Bronze vs. Silver) + Preview Gold AEON 360
WITH checks AS (
  SELECT
    'Fact_EP_Judge -> silver_ep_underwriting' AS pipeline_flow,
    'FIN_AMT (Financed Principal MYR)' AS control_metric,
    (SELECT COUNT(*) FROM `acsm_bronze.Fact_EP_Judge`) AS bronze_rows,
    (SELECT COUNT(*) FROM `acsm_silver.silver_ep_underwriting`) AS silver_rows,
    (SELECT ROUND(SUM(CAST(FIN_AMT AS NUMERIC)), 2) FROM `acsm_bronze.Fact_EP_Judge`) AS bronze_total_myr,
    (SELECT ROUND(SUM(FIN_AMT), 2) FROM `acsm_silver.silver_ep_underwriting`) AS silver_total_myr
  UNION ALL
  SELECT
    'Fact_CC_Judge -> silver_cc_underwriting' AS pipeline_flow,
    'B_CrLimit (Approved Credit Limit MYR)' AS control_metric,
    (SELECT COUNT(*) FROM `acsm_bronze.Fact_CC_Judge`) AS bronze_rows,
    (SELECT COUNT(*) FROM `acsm_silver.silver_cc_underwriting`) AS silver_rows,
    (SELECT ROUND(SUM(CAST(B_CrLimit AS NUMERIC)), 2) FROM `acsm_bronze.Fact_CC_Judge`) AS bronze_total_myr,
    (SELECT ROUND(SUM(B_CrLimit), 2) FROM `acsm_silver.silver_cc_underwriting`) AS silver_total_myr
  UNION ALL
  SELECT
    'Fact_EP_Collection + Fact_CC_Collection -> silver_collections_summary' AS pipeline_flow,
    'Unpaid_OSP (Combined Unpaid Principal MYR)' AS control_metric,
    (SELECT COUNT(DISTINCT CIF_No) FROM (
      SELECT CIF_No FROM `acsm_bronze.Fact_EP_Collection`
      UNION DISTINCT
      SELECT CIF_No FROM `acsm_bronze.Fact_CC_Collection`
    )) AS bronze_rows,
    (SELECT COUNT(*) FROM `acsm_silver.silver_collections_summary`) AS silver_rows,
    (SELECT ROUND(SUM(CAST(Unpaid_OSP AS NUMERIC)), 2) FROM `acsm_bronze.Fact_EP_Collection`) +
      (SELECT ROUND(SUM(CAST(Unpaid_OSP AS NUMERIC)), 2) FROM `acsm_bronze.Fact_CC_Collection`) AS bronze_total_myr,
    (SELECT ROUND(SUM(combined_unpaid_osp), 2) FROM `acsm_silver.silver_collections_summary`) AS silver_total_myr
)
SELECT
  pipeline_flow,
  control_metric,
  bronze_rows,
  silver_rows,
  bronze_total_myr,
  silver_total_myr,
  (silver_total_myr - bronze_total_myr) AS variance_myr,
  IF(bronze_rows = silver_rows AND ABS(silver_total_myr - bronze_total_myr) = 0, 'PASS (0.00 MYR VARIANCE)', 'INVESTIGATE') AS audit_status
FROM checks;

In [ ]:
%%bigquery --project $PROJECT_ID --location $LOCATION
-- 3. Preview Top 10 Customers in the Gold AEON 360 Feature Store (`acsm_gold.gold_aeon360_customer_profile`)
SELECT
  CIF_ID,
  CIF_NM,
  State,
  B_NetIncome,
  ep_app_count,
  total_ep_financed_myr,
  cc_app_count,
  total_cc_limit_myr,
  latest_ctos_score,
  combined_unpaid_osp,
  worst_collection_score_grade,
  active_card_count,
  total_cp_usage_myr
FROM `acsm_gold.gold_aeon360_customer_profile`
ORDER BY total_ep_financed_myr + total_cc_limit_myr DESC
LIMIT 10;